In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Data Loading

In [ ]:
# Define the file paths for the training and test data
train_path = '/kaggle/input/house-prices-advanced-regression-techniques/train.csv'
test_path = '/kaggle/input/house-prices-advanced-regression-techniques/test.csv'

# Load the CSV files into pandas DataFrames
try:
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Display the first five rows of the training DataFrame to verify it loaded correctly
    print("First 5 rows of the training data:")
    print(train_df.head())

except FileNotFoundError:
    print("Error: Make sure the file paths are correct.")
    print(f"Attempted to load train data from: {train_path}")
    print(f"Attempted to load test data from: {test_path}")


# EDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Define the file paths for the training and test data
train_path = '/kaggle/input/house-prices-advanced-regression-techniques/train.csv'
test_path = '/kaggle/input/house-prices-advanced-regression-techniques/test.csv'

# Load the CSV files into pandas DataFrames
try:
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # --- Basic Data Exploration ---
    print("--- Data Info ---")
    print("This shows the data types and non-null counts for each column.")
    train_df.info()

    print("\n--- Descriptive Statistics ---")
    print("This provides a statistical summary for all numerical columns.")
    print(train_df.describe())


    # --- 1. Distribution of SalePrice ---
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Plot raw SalePrice
    sns.histplot(train_df['SalePrice'], kde=True, ax=axes[0], color='blue')
    axes[0].set_title('Distribution of SalePrice (Raw)')
    axes[0].set_xlabel('Sale Price')
    axes[0].set_ylabel('Frequency')

    # Plot log-transformed SalePrice
    sns.histplot(np.log1p(train_df['SalePrice']), kde=True, ax=axes[1], color='green')
    axes[1].set_title('Distribution of SalePrice (Log-Transformed)')
    axes[1].set_xlabel('Log(1 + Sale Price)')
    axes[1].set_ylabel('Frequency')

    fig.suptitle('SalePrice Distribution Analysis', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


    # --- 2. Correlation Heatmap of Top Numeric Features ---
    # Calculate correlations with SalePrice
    correlations = train_df.corr(numeric_only=True)
    # Get the top 10 features most correlated with SalePrice
    top_10_cols = correlations.nlargest(10, 'SalePrice')['SalePrice'].index
    top_10_corr_matrix = train_df[top_10_cols].corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(top_10_corr_matrix, annot=True, cmap='viridis', fmt='.2f', linewidths=0.5)
    plt.title('Correlation Heatmap of Top 10 Features vs. SalePrice', fontsize=16)
    plt.show()


    # --- 3. Boxplots for Categorical Variables ---
    # Boxplot for Neighborhood vs SalePrice
    plt.figure(figsize=(18, 8))
    sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df)
    plt.xticks(rotation=45, ha='right')
    plt.title('SalePrice Distribution by Neighborhood', fontsize=16)
    plt.xlabel('Neighborhood')
    plt.ylabel('Sale Price')
    plt.tight_layout()
    plt.show()

    # Boxplot for Overall Quality vs SalePrice
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='OverallQual', y='SalePrice', data=train_df)
    plt.title('SalePrice Distribution by Overall Quality', fontsize=16)
    plt.xlabel('Overall Quality')
    plt.ylabel('Sale Price')
    plt.tight_layout()
    plt.show()


except FileNotFoundError:
    print("Error: Make sure the file paths are correct.")
    print(f"Attempted to load train data from: {train_path}")
    print(f"Attempted to load test data from: {test_path}")
except Exception as e:
    print(f"An error occurred: {e}")

# Split Data

In [ ]:
    from sklearn.model_selection import train_test_split

    # Separate features (X) and target (y)
    X = train_df.drop('SalePrice', axis=1)
    y = train_df['SalePrice']

    # Perform an 80/20 split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("\n--- Train-Test Split Shapes ---")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

# Train Linear Regression Model

In [ ]:
    # --- 5. Model Training: Linear Regression ---
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, r2_score
    from sklearn.impute import SimpleImputer

    # --- Preprocessing ---
    # Select only numeric columns for this simple model
    X_train_numeric = X_train.select_dtypes(include=np.number)
    X_test_numeric = X_test.select_dtypes(include=np.number)

    # Impute missing values using the median of the training data
    # This avoids data leakage from the test set
    imputer = SimpleImputer(strategy='median')
    X_train_imputed = imputer.fit_transform(X_train_numeric)
    X_test_imputed = imputer.transform(X_test_numeric)

    # Convert imputed data back to DataFrame to keep column names
    X_train_imputed = pd.DataFrame(X_train_imputed, columns=X_train_numeric.columns)
    X_test_imputed = pd.DataFrame(X_test_imputed, columns=X_test_numeric.columns)

    print("\n--- Preprocessing Complete ---")
    print(f"Number of numeric features used: {X_train_imputed.shape[1]}")


    # --- Model Training ---
    print("\n--- Training Linear Regression Model ---")
    model = LinearRegression()
    model.fit(X_train_imputed, y_train)
    print("Model training complete.")


    # --- Evaluation ---
    print("\n--- Evaluating Model Performance ---")
    y_pred = model.predict(X_test_imputed)

    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
    print(f"R-squared (R2 Score): {r2:.4f}")

# Feature Engineering

In [ ]:
# --- Feature Engineering Function ---
def engineer_features(df):
    # Explicitly create a copy to avoid SettingWithCopyWarning and ensure changes stick
    df = df.copy()
    # Fill NA values for features used in new feature creation to avoid NaN propagation
    for col in ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'FullBath', 'HalfBath', 'BsmtFullBath', 'BsmtHalfBath']:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    
    # Handle potential negative ages if data is inconsistent
    df['HouseAge'] = df['HouseAge'].apply(lambda x: max(x, 0))
    df['RemodAge'] = df['RemodAge'].apply(lambda x: max(x, 0))
    return df

print("\n--- Engineering New Features ---")
train_df = engineer_features(train_df)
# We will apply the same transformation to the actual test data later if needed
# test_df = engineer_features(test_df)

# --- Data Splitting and Preprocessing ---
print("\n--- Splitting Data and Preprocessing ---")
X = train_df.drop('SalePrice', axis=1)
y = np.log1p(train_df['SalePrice']) # Use log-transformed target

numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(include=['object']).columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify feature types
numeric_features = X_train.select_dtypes(include=np.number).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

# --- Preprocessing Pipeline ---
# 1. Impute missing numeric values using the median
imputer = SimpleImputer(strategy='median')
X_train[numeric_features] = imputer.fit_transform(X_train[numeric_features])
X_test[numeric_features] = imputer.transform(X_test[numeric_features])

# 2. One-Hot Encode categorical variables
X_train = pd.get_dummies(X_train, columns=categorical_features, dummy_na=True)
X_test = pd.get_dummies(X_test, columns=categorical_features, dummy_na=True)

# 3. Align columns between train and test sets
# Use reindex to ensure the test set has exactly the same columns as the training set
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


# --- Model Training ---
print("\n--- Training Linear Regression Model ---")
model = LinearRegression()
model.fit(X_train, y_train)
print("Model training complete.")


# --- Model Evaluation ---
print("\n--- Evaluating Model Performance ---")
# Predictions will be on the log scale, so we need to transform them back
y_pred_log = model.predict(X_test)

# Clip predictions to prevent overflow errors from extreme values
y_pred_log = np.clip(y_pred_log, None, 700) 

y_pred = np.expm1(y_pred_log)
y_test_orig = np.expm1(y_test) # Original scale y_test for comparison

# Create a mask to filter out non-finite values from both arrays before scoring
finite_mask = np.isfinite(y_test_orig) & np.isfinite(y_pred)

if np.sum(finite_mask) == 0:
    print("Error: No finite values available to calculate metrics. Check for NaNs in predictions or test data.")
else:
    # Calculate metrics on the original price scale using only the filtered, finite values
    rmse = np.sqrt(mean_squared_error(y_test_orig[finite_mask], y_pred[finite_mask]))
    r2 = r2_score(y_test_orig[finite_mask], y_pred[finite_mask])
    
    print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
    print(f"R-squared (R2) Score: {r2:.4f}")


# Train and compare models

In [ ]:
# --- Train and Compare Advanced Models ---
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
print("\n--- Training and Comparing Advanced Models ---")

# Define models to train
models = {
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(random_state=42, n_jobs=-1)
}

results = {}

# Loop through models, train, predict, and evaluate
for name, model_instance in models.items():
    print(f"\n--- Training {name} ---")
    model_instance.fit(X_train, y_train)
    
    y_pred_log_adv = model_instance.predict(X_test)
    y_pred_adv = np.expm1(y_pred_log_adv)
    
    # Use the same finite mask for a fair comparison
    finite_mask_adv = np.isfinite(y_test_orig) & np.isfinite(y_pred_adv)
    
    if np.sum(finite_mask_adv) > 0:
        rmse_adv = np.sqrt(mean_squared_error(y_test_orig[finite_mask_adv], y_pred_adv[finite_mask_adv]))
        results[name] = rmse_adv
    else:
        results[name] = np.inf # Assign infinity if calculation fails
        print(f"Could not calculate RMSE for {name} due to non-finite values.")

# --- Display Final Comparison ---
print("\n--- Model Comparison (Validation RMSE) ---")
for name, rmse_val in sorted(results.items(), key=lambda item: item[1]):
    print(f"{name}: ${rmse_val:,.2f}")


# Best Model and Submission File

In [ ]:
# --- Display Final Comparison ---
print("\n--- Model Comparison (Validation RMSE) ---")
for name, rmse_val in sorted(results.items(), key=lambda item: item[1]):
    print(f"{name}: ${rmse_val:,.2f}")

# --- Create Submission File with Best Model ---
print("\n--- Generating Submission File ---")

# 1. Identify the best model from the results
best_model_name = min(results, key=results.get)
print(f"The best model is: {best_model_name}")

# 2. Retrain the best model on the FULL dataset
print(f"\n--- Retraining {best_model_name} on the full dataset ---")

# Start with fresh, full data
X_full = train_df.drop('SalePrice', axis=1)
y_full = np.log1p(train_df['SalePrice'])
test_full = test_df.copy()

# Get the IDs for the submission file before any processing alters the dataframe
submission_ids = test_full['Id']

# Align columns BEFORE any processing. This ensures test_full has all the columns
# from X_full, adding any missing ones as NaN, which the imputer will handle.
X_full, test_full = X_full.align(test_full, join='left', axis=1)

# Preprocess the full training and test data consistently
imputer_full = SimpleImputer(strategy='median')
X_full[numeric_features] = imputer_full.fit_transform(X_full[numeric_features])
test_full[numeric_features] = imputer_full.transform(test_full[numeric_features])

X_full = pd.get_dummies(X_full, columns=categorical_features, dummy_na=True)
test_full = pd.get_dummies(test_full, columns=categorical_features, dummy_na=True)

# Align all columns
X_full, test_full = X_full.align(test_full, join='left', axis=1, fill_value=0)
# Also drop any columns that might be in test but not train
test_full = test_full[X_full.columns]


# Create a new instance of the best model
final_model = models[best_model_name]
final_model.fit(X_full, y_full)

# 4. Make final predictions
final_predictions_log = final_model.predict(test_full)
final_predictions = np.expm1(final_predictions_log)

# 5. Create submission DataFrame
submission_df = pd.DataFrame({'Id': submission_ids, 'SalePrice': final_predictions})

# 6. Save to CSV
submission_df.to_csv('submission.csv', index=False)
print("\n'submission.csv' created successfully!")
print(submission_df.head())